# RecSys МАГОЛЕГО, ФКН ВШЭ

## Домашняя работа 2 (Часть 1): Graph Laplacian

### Оценивание и штрафы
Всего заданий: **7**, максимальная оценка — **6 баллов (+1.5 бонус)**.

Задание выполняется самостоятельно. «Похожие» решения считаются плагиатом и все задействованные студенты (в том числе те, у кого списали) не могут получить за него больше 0 баллов. Весь код должен быть написан самостоятельно. Чужим кодом пользоваться запрещается,даже с указанием ссылки на источник. В разумных рамках, конечно. Взять пару очевидных строчек кода для реализации какого-то небольшого функционала можно.

Неэффективная реализация кода может негативно отразиться на оценке (например, лишние циклы, `np.vectorize`, `np.apply_along_axis` и так далее). Также оценка может быть снижена за плохо читаемый код и плохо оформленные графики. Все ответы должны сопровождаться кодом или комментариями о том, как они были получены.

Использование генеративных языковых моделей разрешено только в случае явного указания на это. Необходимо прописать (в соответствующих пунктах, где использовались, либо в начале/конце работы):

- какая языковая модель использовалась
- какие использовались промпты и в каких частях работы
- с какими сложностями вы столкнулись при использовании генеративных моделей, с чем они помогли больше всего
  
Copy-paste ответа генеративной модели запрещается (кроме графиков, но все равно нужно явно прописывать использование)

**Дедлайн: 15.03.2026 23:59 (по МСК)**

**Сдавать сюда: [Классрум](https://classroom.google.com/c/ODQzNjI1ODIzMDEy/a/ODQ2ODc0NzExMTI5/details)**

### О задании
В данном домашнем задании вы реализуете алгоритмы рекомендаций на Graph Laplacian для построения эмбеддингов и обучите supervised модель на датасете Yahoo! Movies.

P.S Пожалуйста, аккуратно оформляйте графики, ориентироваться можно на [это](https://github.com/esokolov/ml-course-hse/blob/master/2022-fall/seminars/sem02-charts.ipynb). У графиков обязательно должно быть:

- Название
- Подписанные оси
- Легенда, если необходимо (например, если несколько графиков на одном рисунке)
- Все должно быть четко видно и ничего не сливаться
- За некрасивые графики можем снизить баллы

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

## Структура датасета

Датасет можете скачать по [ссылке](https://drive.google.com/file/d/1PsAL83MQnvQuTpjrNXs8CiPpOeSGcUL-/view?usp=share_link)

Датасет состоит из 6 основных файлов. Все данные представлены в текстовом формате, где колонки разделены символом табуляции `\t`.

### 1. Оценки пользователей
В этих файлах содержатся оценки фильмов пользователей
-  **Файлы:** `ydata-ymovies-user-movie-ratings-train-v1_0` и `ydata-ymovies-user-movie-ratings-test-v1_0`
- **Поля:**
  * User ID
  * Movie ID
  * Rating_13: от 1 (F) до 13 (A+)
  * Rating_5: упрощенная шкала от 1 до 5



### 2. Демография пользователей
Информация про аудиторию

- **Файл:** `ydata-ymovies-user-demographics-v1_0`
- **Поля:** 
  * User ID
  * Year of birth
  * Sex (`m`/`f`)

### 3. Описание фильмов
Метаданные фильмов
* **Файл:** `ydata-ymovies-movie-content-descr-v1_0`
* **Что внутри:** Название, синопсис, жанры, режиссеры, актеры, количество наград, средняя оценка критиков и даже ссылки на постеры.
* **Важно:** Если данных нет, стоит заглушка `\N`. Если в поле несколько значений (например, список актеров), они разделены символом `|`.

### 4. Списки соответствия 
Файлы `mapping-to-movielens` и `mapping-to-eachmovie` позволяют связать ID фильмов Yahoo с другими популярными датасетами (MovieLens и EachMovie). Это полезно, если вы хотите объединить данные из разных источников


### Graph Laplacian

В этой части мы будем использовать строить эмбеддинги пользователей и фильмов с помощью графа Лапласиана, чтобы потом на этих эмбеддингах обучать supervised модель, которая уже будет рекомендовать фильмы

**Задание 0 (1 балл):** Загрузите необходимые данные и постройте неориентированный невзвешенный двудольный граф. Ребро между пользователем $u$ и фильмом $i$ существует, если пользователь поставил фильму любую оценку. Сформируйте матрицу смежности $A$ (не забудьте использовать разреженные матрицы)

In [ ]:
# Your code her (￣～￣;)

**Задание 1 (1 балл):** На основе матрицы $A$ вычислите диагональную матрицу степеней $D$ и постройте нормализованный Лапласиан граф:

$$
L_{sym} = I - D^{-1/2} A D^{-1/2}
$$

Найдите $d$ (например, $d=64$) собственных вектора матрицы $L_{sym}$, соответствующих ее наименьшим собственным значениям. Не забудьте убрать самый первый вектор (почему?). Полученные вектора будем использовать в качестве признаков

In [ ]:
# Your code her (￣～￣;)

**Задание 2 (0.5 балла):** Модель должна понимать, какие фильмы релевантны, а какие нет. Поэтому необходимо также добавить к существующим тренировочным данным примеры нулевого класса (negative sampling)

Сгенерируйте негативные примеры (класс 0), случайно выбирая пары «пользователь-фильм», которых нет в тренировочных данных. Рекомендуемая пропорция негативных к позитивным примерам — 2:1 или 3:1

Создайте итоговую матрицу признаков $X$. Для каждой пары $(u,i)$ сформируйте вектор признаков, объединив эмбеддинг пользователя и эмбеддинг фильма (например, через конкатенацию). Вектор ответов y должен состоять из бинарных меток (1 и 0).

In [ ]:
# Your code her (￣～￣;)

**Задание 3 (1 балл):** Пришло время обучить модель на наших признаках:

- Выберите межу `RandomForestClassifier` и `GradientBoostingClassifier`
- Предсказывайте вероятность, что фильм понравится пользователю
- Провалидируйте вашу модель на тестовом датасете с метриками из ДЗ-1

In [ ]:
# Your code her (￣～￣;)

**Задание 4 (1 балл):** Добавьте к вашим графовым признакам метаданные из файлов `ydata-ymovies-user-demographics` и `ydata-ymovies-movie-content-descr`. Замерьте, насколько изменились метрики после обогащения модели этими признаками

In [ ]:
# Your code her (￣～￣;)

**Задание 5:** Давайте сделаем нашу модель более продвинутой и точной. Для этого вам предстоит реализовать следующие улучшения:

**1. Взвешенный граф взаимодействий (0.5 балла)**

Сейчас мы рассматриваем только сам факт взаимодействия (оценил / не оценил). Однако значения оценок несут в себе много полезной информации!

Вместо единиц в матрице смежности $A$ используйте нормализованные значения из колонки с рейтингами

**2. Сэмплинг «сложных» негативных примеров (Hard Negative Sampling) (0.5 балла)**

Генерация случайных негативных примеров (Uniform Negative Sampling) — это хороший старт. Но случайный несмотренный фильм может оказаться никому не известным артхаусом, и модели будет слишком легко отличить его от популярных просмотренных блокбастеров

Реализуйте сэмплинг сложных негативных примеров. Например, предлагайте в качестве нулей фильмы, которые очень популярны во всем датасете, но данный конкретный пользователь их так и не посмотрел

**3. Эксперименты с объединением эмбеддингов (0.5 балла)**

То, как именно мы объединяем эмбеддинг пользователя $E_u$ и фильма $E_i$, критически влияет на качество обучения модели

Сравните разные способы формирования итогового вектора признаков для пары $(u, i)$:
  - Конкатенация: $[E_u, E_i]$
  - Поэлементное умножение (Hadamard product): $E_u \odot E_i$
  - Косинусная близость: Скалярное произведение нормализованных векторов

На каждый пункт дайте ответ, как изменились метрики рекомендаций / качество собственных векторов? Без анализа баллы не засчитаются!

In [ ]:
# Your code her (￣～￣;)

**Бонусное задание 1 (1.5 балла):** Представьте, что появился новый пользователь или в каталог добавили премьеру фильма. Поскольку их не было в обучающих данных, у них нет связей в графе, а значит, мы не можем вычислить для них векторы Лапласа

Как сделать для них качественные рекомендации, опираясь только на файлы `ydata-ymovies-user-demographics` и `ydata-ymovies-movie-content-descr`? Предложите 1-2 способа и проверьте их на тестовой выборке.

In [ ]:
# Your code her (￣～￣;)

## Conclusion

Опишите выводы ваших экспериментов:
